In [1]:
import destruction_models as models
from tensorflow.keras import callbacks, metrics
from tensorflow.keras.utils import Sequence
from destruction_utilities import *
import tensorflow as tf
import time
import pickle

In [2]:
CITY = 'aleppo'
BATCH_SIZE = 32
TILE_SIZE = [128,128]

In [3]:
# class SiameseGenerator(Sequence):
#     def __init__(self, images, labels, batch_size=BATCH_SIZE):
#         self.images_t0 = images[0]
#         self.images_tt = images[1]
#         self.labels = labels
#         self.batch_size = batch_size
#         self.tuple_pairs = make_tuple_pair(self.images_t0.shape[0], self.batch_size)
#         np.random.shuffle(self.tuple_pairs)
   
#     def __len__(self):
#         return len(self.images_t0)//self.batch_size
    
#     def __getitem__(self, index):

#         index_range = self.tuple_pairs[index]
#         indices = np.arange(0,32)
#         np.random.shuffle(indices)
        
#         X_t0 = self.images_t0[index_range[0]:index_range[1]][indices]
#         X_t1 = self.images_tt[index_range[0]:index_range[1]][indices]
#         y = self.labels[index_range[0]:index_range[1]][indices]
        
#         return {'images_t0':X_t0/255.0, 'images_tt':X_t1/255.0}, y

class SiameseGenerator(Sequence):
    def __init__(self, images, labels, batch_size=BATCH_SIZE):
        self.images_t0 = images[0]
        self.images_tt = images[1]
        self.labels = labels
        self.batch_size = batch_size
        
    def __len__(self):
        return len(self.images_t0)//self.batch_size
    
    def __getitem__(self, index):

        X_t0 = self.images_t0[index*self.batch_size:(index+1)*self.batch_size]
        X_t1 = self.images_tt[index*self.batch_size:(index+1)*self.batch_size]
        y = self.labels[index*self.batch_size:(index+1)*self.batch_size]
        
        return {'images_t0':X_t0, 'images_tt':X_t1}, y

In [4]:
train_images_t0 = read_zarr(CITY, 'images_siamese_train_t0')
train_images_tt = read_zarr(CITY, 'images_siamese_train_tt')
train_labels = read_zarr(CITY, 'labels_siamese_train')

valid_images_t0 = read_zarr(CITY, 'images_siamese_valid_t0')
valid_images_tt = read_zarr(CITY, 'images_siamese_valid_tt')
valid_labels = read_zarr(CITY, 'labels_siamese_valid')

train_gen = SiameseGenerator((train_images_t0, train_images_tt), train_labels)
valid_gen = SiameseGenerator((valid_images_t0, valid_images_tt), valid_labels)


In [5]:
def run_model(train_images, train_labels, valid_images, valid_labels):
    train_gen = SiameseGenerator((train_images[0], train_images[1]), train_labels)
    valid_gen = SiameseGenerator((valid_images[0], valid_images[1]), valid_labels)
    
    training_callbacks = [
        callbacks.EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True)
    ]
    
    filters = random.choice([32])
    dropout = random.choice(np.linspace(0.1, 0.2))
    epochs = random.choice(np.arange(10,20))
    units = random.choice([24, 32])
    lr = random.choice([0.003, 0.01, 0.03])

    args_encode  = dict(filters=filters, dropout=dropout) # ! Check parameters before run
    args_dense  = dict(units=units, dropout=dropout) # ! Check parameters before run

    parameters = f'Model parameters: filters={filters}, dropout={dropout}, epochs={epochs}, units={units}, learning_rate={lr}'
    print(parameters)
    
#     model = models.siamese_convolutional_network(shape=(*TILE_SIZE, 3), args_encode=args_encode, args_dense=args_dense)
    model = models.siamese_convolutional_network(shape=(*TILE_SIZE, 3),  args_encode=dict(filters=8, dropout=0), args_dense=dict(units=16, dropout=0))
   
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy',metrics.AUC(num_thresholds=200, curve='ROC', name='auc')])
    model.summary()
    

    # Train model on dataset
    history = model.fit_generator(
        train_gen,
        validation_data=valid_gen,
        epochs=epochs,
        verbose=1,
        callbacks=training_callbacks
    )
    
    return model, history, parameters

In [6]:
for i in range(0,5):
    m = run_model((train_images_t0, train_images_tt), train_labels, (valid_images_t0, valid_images_tt), valid_labels)
    model = m[0]
    history = m[1]
    parameters = m[2]
    print("Model optimization complete..\n\n")
    ts = str(time.time())
    with open(f'../models/{CITY}_SNN_RUN{i}_{ts}_hist', 'wb') as file_pi:
        pickle.dump(history.history, file_pi)
    
    model.save(f'../models/{CITY}_SNN_RUN{i}_{ts}', save_format="h5")
    
    with open('../models/run_parameters.txt', "a") as file:
        file.write(f'{CITY}_SNN_RUN{i}_{ts}: \n \t{parameters}\n')


Model parameters: filters=32, dropout=0.14489795918367349, epochs=15, units=32, learning_rate=0.03
Metal device set to: Apple M1

systemMemory: 16.00 GB
maxCacheSize: 5.33 GB

Model: "siamese_convolutional_network"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 images_t0 (InputLayer)         [(None, 128, 128, 3  0           []                               
                                )]                                                                
                                                                                                  
 images_tt (InputLayer)         [(None, 128, 128, 3  0           []                               
                                )]                                                                
                                                                                            

2022-07-10 23:06:44.072923: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2022-07-10 23:06:44.073026: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/var/folders/z1/7pydgng971nfzdf2rwg_p__r0000gn/T/ipykernel_93203/7179167.py:30: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history = model.fit_generator(


Epoch 1/15


2022-07-10 23:06:44.435415: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2022-07-10 23:06:45.222356: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


3210/3210 [==============================] - ETA: 0s - loss: 0.1567 - accuracy: 0.9625 - auc: 0.6391

2022-07-10 23:16:56.262064: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


3210/3210 [==============================] - 663s 206ms/step - loss: 0.1567 - accuracy: 0.9625 - auc: 0.6391 - val_loss: 0.1295 - val_accuracy: 0.9689 - val_auc: 0.7681
Epoch 2/15
 547/3210 [====>.........................] - ETA: 8:46 - loss: 0.1753 - accuracy: 0.9551 - auc: 0.6773

KeyboardInterrupt: 

```
filters=32, dropout=0.13163265306122449, epochs=77, units=32, learning_rate=0.01
filters=32, dropout=0.18571428571428572, epochs=88, units=48, learning_rate=0.003
```